In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
df = pd.read_csv("D:\Projects-Guvi\EV\ChargingRecords.csv")

<>:1: SyntaxWarning: invalid escape sequence '\P'
<>:1: SyntaxWarning: invalid escape sequence '\P'
C:\Users\Jyots\AppData\Local\Temp\ipykernel_17680\2511024403.py:1: SyntaxWarning: invalid escape sequence '\P'
  df = pd.read_csv("D:\Projects-Guvi\EV\ChargingRecords.csv")


In [3]:
df.describe()

,UserID,ChargerID,ChargerCompany,ChargerType,Duration,Demand
count,72856.000000,72856.000000,72856.000000,72856.00000,72856.000000,72856.000000
mean,338.164667,525.489212,0.627443,0.20524,151.194054,17.444445
std,553.970996,569.544469,0.483489,0.40388,146.430286,13.492728
min,0.000000,1.000000,0.000000,0.00000,-29.000000,0.010000
25%,0.000000,107.000000,0.000000,0.00000,40.000000,7.530000
50%,62.000000,301.000000,1.000000,0.00000,112.000000,14.100000
75%,407.000000,738.000000,1.000000,0.00000,202.000000,23.200000
max,2501.000000,2671.000000,1.000000,1.00000,1573.000000,97.000000


In [4]:
df.drop_duplicates(inplace=True)

In [5]:
df = df.astype({
    'StartDay': 'datetime64[ns]',
    'StartTime': 'datetime64[ns]',
    'EndDay': 'datetime64[ns]',
    'EndTime': 'datetime64[ns]',
    'StartDatetime': 'datetime64[ns]',
    'EndDatetime': 'datetime64[ns]'
})

In [6]:
# Checking the Multicollinearity among the independent variables

# Select only numeric, independent features for analysis
# Exclude unique IDs (like UserID, ChargerID) and your target variable (like Demand)
features = df[['ChargerCompany', 'ChargerType', 'Duration']]

# Calculate the correlation matrix
corr_matrix = features.corr()
print(corr_matrix)

                ChargerCompany  ChargerType  Duration
ChargerCompany        1.000000    -0.659472  0.246896
ChargerType          -0.659472     1.000000 -0.399730
Duration              0.246896    -0.399730  1.000000


In [7]:
corr_inverse = np.linalg.inv(corr_matrix.values)
vif_values = np.diag(corr_inverse)
vif_data = pd.DataFrame({'Feature': features.columns,'VIF': vif_values
})

print(vif_data.sort_values(by="VIF", ascending=False).reset_index(drop=True))

          Feature       VIF
0     ChargerType  1.978917
1  ChargerCompany  1.770653
2        Duration  1.190870


In [8]:
df['Demand'].median()

np.float64(14.1)

In [9]:
# 2. Define the bin edges and the labels
# We use -float('inf') and float('inf') to capture all numbers below and above the median
bins = [-float ('inf'), 14.1, float('inf')]

labels = ["Normal_Demand","High_Demand"]

# 3. Create the new column
df['Demand'] = pd.cut(df['Demand'], bins=bins, labels=labels)


In [52]:
df.head(2)

,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand
0,0,1,1,hotel,0,2022-09-15,2026-10-28 20:54:02,2022-09-15,2026-10-28 23:59:13,2022-09-15 20:54:00,2022-09-15 23:59:00,185,High_Demand
1,0,1,1,hotel,0,2022-09-14,2026-10-28 20:01:05,2022-09-14,2026-10-28 21:31:04,2022-09-14 20:01:00,2022-09-14 21:31:00,90,Normal_Demand


In [55]:
df['Demand'].value_counts()

Demand
Normal_Demand    36448
High_Demand      36395
Name: count, dtype: int64

In [90]:
X = df [["UserID","ChargerID","ChargerCompany","ChargerType", "Duration"]]

In [91]:
Y = df ['Demand']

In [92]:
# splitting data into train and test set

from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X,Y, test_size=1/3)

In [93]:
from sklearn.preprocessing import StandardScaler


# Initialize StandardScaler
ss = StandardScaler()

# Fit and transform the training data
X_train = ss.fit_transform(X_train)

# Transform the test data
X_test = ss.transform(X_test)


# Logistic Regression

In [94]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


log = LogisticRegression()
log.fit(X_train,Y_train)
train_score = log.score(X_train,Y_train)

#Predections

Y_pred = log.predict(X_test)

Y_prob = log.predict_proba(X_test)[:, 1]

# Calculate the ROC-AUC score
roc_auc = roc_auc_score(Y_test, Y_prob)


In [95]:
from sklearn.metrics import classification_report


print('Logistic regression \n ')
print(classification_report(Y_test,Y_pred))
print(f"ROC-AUC Score: {roc_auc:.4f}")

Logistic regression 
 
               precision    recall  f1-score   support

  High_Demand       0.82      0.88      0.85     12079
Normal_Demand       0.87      0.81      0.84     12202

     accuracy                           0.84     24281
    macro avg       0.85      0.84      0.84     24281
 weighted avg       0.85      0.84      0.84     24281

ROC-AUC Score: 0.9140


# Decision Tree

In [97]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

dtree = DecisionTreeClassifier()
dtree.fit(X_train,Y_train)

train_score = dtree.score(X_train,Y_train)

Y_pred = dtree.predict(X_test)

# Get the predicted probabilities for the test set
# [:, 1] extracts the probabilities for the positive class (e.g., 'Normal_Demand' or 'High_Demand')
Y_prob = dtree.predict_proba(X_test)[:, 1]

# Calculate the ROC-AUC score
roc_auc = roc_auc_score(Y_test, Y_prob)

In [98]:
print('Decison Tree \n')
print(classification_report(Y_test,Y_pred))
print(f"ROC-AUC Score: {roc_auc:.4f}")

Decison Tree 

               precision    recall  f1-score   support

  High_Demand       0.91      0.92      0.92     12079
Normal_Demand       0.92      0.91      0.92     12202

     accuracy                           0.92     24281
    macro avg       0.92      0.92      0.92     24281
 weighted avg       0.92      0.92      0.92     24281

ROC-AUC Score: 0.9195


# Random Forest

In [99]:
from sklearn.ensemble import RandomForestClassifier

rforest = RandomForestClassifier()

rforest.fit(X_train,Y_train)

Y_pred = rforest.predict(X_test)

# Get the predicted probabilities for the test set
# [:, 1] extracts the probabilities for the positive class (e.g., 'Normal_Demand' or 'High_Demand')
Y_prob = rforest.predict_proba(X_test)[:, 1]

threshold = 0.7  # Adjust this value based on your business strategy

# Assign labels: 'High_Demand' if probability >= threshold, else 'Normal_Demand'
Y_pred = np.where(Y_prob >= threshold, 'High_Demand', 'Normal_Demand')

# Calculate the ROC-AUC score
roc_auc = roc_auc_score(Y_test, Y_prob)



In [100]:
print('Random Forest \n')
print(classification_report(Y_test,Y_pred))
print(f"ROC-AUC Score: {roc_auc:.4f}")


print("--- Total Summary Counts ---")
unique, counts = np.unique(Y_pred, return_counts=True)
for status, count in zip(unique, counts):
    print(f"{status}: {count} rows")

# 2. Prints a quick sneak-peek of the first 10 individual predictions
print("\n--- First 10 Sample Predictions ---")
for i in range(min(10, len(Y_pred))):
    print(f"Row {i}: Predicted Status -> {Y_pred[i]} (High Demand Probability: {Y_prob[i]:.2%})")

Random Forest 

               precision    recall  f1-score   support

  High_Demand       0.03      0.03      0.03     12079
Normal_Demand       0.11      0.12      0.11     12202

     accuracy                           0.07     24281
    macro avg       0.07      0.07      0.07     24281
 weighted avg       0.07      0.07      0.07     24281

ROC-AUC Score: 0.9780
--- Total Summary Counts ---
High_Demand: 11102 rows
Normal_Demand: 13179 rows

--- First 10 Sample Predictions ---
Row 0: Predicted Status -> High_Demand (High Demand Probability: 100.00%)
Row 1: Predicted Status -> Normal_Demand (High Demand Probability: 0.00%)
Row 2: Predicted Status -> Normal_Demand (High Demand Probability: 28.00%)
Row 3: Predicted Status -> Normal_Demand (High Demand Probability: 14.68%)
Row 4: Predicted Status -> Normal_Demand (High Demand Probability: 31.00%)
Row 5: Predicted Status -> Normal_Demand (High Demand Probability: 0.00%)
Row 6: Predicted Status -> Normal_Demand (High Demand Probability:

# KNN

In [101]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier()

knn.fit(X_train,Y_train)

Y_pred = knn.predict(X_test)

# Get the predicted probabilities for the test set
# [:, 1] extracts the probabilities for the positive class (e.g., 'Normal_Demand' or 'High_Demand')
Y_prob = knn.predict_proba(X_test)[:, 1]

# Calculate the ROC-AUC score
roc_auc = roc_auc_score(Y_test, Y_prob)

In [102]:
print('KNN \n')
print(classification_report(Y_test,Y_pred))
print(f"ROC-AUC Score: {roc_auc:.4f}")

KNN 

               precision    recall  f1-score   support

  High_Demand       0.90      0.93      0.92     12079
Normal_Demand       0.93      0.90      0.91     12202

     accuracy                           0.91     24281
    macro avg       0.92      0.91      0.91     24281
 weighted avg       0.92      0.91      0.91     24281

ROC-AUC Score: 0.9603


# SVM

In [103]:
from sklearn.svm import SVC

svc = SVC()

svc.fit(X_train,Y_train)

Y_pred = svc.predict(X_test)

roc_auc = roc_auc_score(Y_test, Y_prob)

In [104]:
print('SVM \n')
print(classification_report(Y_test,Y_pred))
print(f"ROC-AUC Score: {roc_auc:.4f}")


SVM 

               precision    recall  f1-score   support

  High_Demand       0.87      0.93      0.90     12079
Normal_Demand       0.92      0.87      0.89     12202

     accuracy                           0.90     24281
    macro avg       0.90      0.90      0.90     24281
 weighted avg       0.90      0.90      0.90     24281

ROC-AUC Score: 0.9603


# XG Boost

In [74]:
import xgboost as xsg

In [70]:
from xgboost import XGBClassifier

xgboost = XGBClassifier()

xgboost.fit(X_train,Y_train)

Y_pred = xgboost.predict(X_test)

ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1], got ['High_Demand' 'Normal_Demand']

# Light GBM

In [105]:
from lightgbm import LGBMClassifier, LGBMRegressor

lgb = LGBMClassifier()

lgb.fit(X_train,Y_train)

Y_pred = lgb.predict(X_test)

# Get the predicted probabilities for the test set
# [:, 1] extracts the probabilities for the positive class (e.g., 'Normal_Demand' or 'High_Demand')
Y_prob = lgb.predict_proba(X_test)[:, 1]

# Calculate the ROC-AUC score
roc_auc = roc_auc_score(Y_test, Y_prob)




[LightGBM] [Info] Number of positive: 24246, number of negative: 24316
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000419 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 771
[LightGBM] [Info] Number of data points in the train set: 48562, number of used features: 5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.499279 -> initscore=-0.002883
[LightGBM] [Info] Start training from score -0.002883


In [106]:
print('Light GBM \n')
print(classification_report(Y_test,Y_pred))

print(f"ROC-AUC Score: {roc_auc:.4f}")

Light GBM 

               precision    recall  f1-score   support

  High_Demand       0.91      0.95      0.93     12079
Normal_Demand       0.95      0.90      0.93     12202

     accuracy                           0.93     24281
    macro avg       0.93      0.93      0.93     24281
 weighted avg       0.93      0.93      0.93     24281

ROC-AUC Score: 0.9767


In [107]:
import pickle

with open('rforest.pkl','wb') as file:
    pickle.dump(rforest,file)

In [108]:
with open('lgb.pkl','wb') as file:
    pickle.dump(lgb,file)